# 02 因果设计：选择方法

## 学习目标

- 理解不同因果推断方法的适用条件
- 根据数据特征选择合适的方法
- 掌握潜在结果框架的核心概念

## 背景

在上一节中，我们发现处理组和对照组存在显著的选择偏差。简单的均值比较不可信。

本节的目标是：基于数据特征，选择最合适的因果推断方法。

---

## 步骤 1：回顾潜在结果框架

对于每个用户 $i$，存在两个潜在结果：
- $Y_i(1)$：领取优惠券后的GMV
- $Y_i(0)$：未领取优惠券的GMV

我们观测到的结果是：
$$Y_i = T_i \cdot Y_i(1) + (1-T_i) \cdot Y_i(0)$$

其中 $T_i$ 是处理指示变量。

**核心问题**：我们只能观测到其中一个潜在结果。

---

## 步骤 2：识别识别策略

根据数据特征，我们可以考虑以下方法：

### 方法 1：随机化实验（RCT）

- **条件**：处理随机分配
- **现状**：我们的数据中处理非随机（平台倾向于向高价值用户发券）
- **结论**：不可行，但可以作为黄金标准参考

### 方法 2：倾向得分匹配（PSM）

- **条件**：可忽略性假设（Ignorability）
- **适用场景**：有丰富协变量，处理组和对照组有重叠
- **我们的数据**：有 historical_gmv、activity_score 等多个协变量，适合PSM

### 方法 3：双重差分（DID）

- **条件**：面板数据，平行趋势假设
- **适用场景**：有处理前后的观测值
- **我们的数据**：有 pre_gmv 和 post_gmv，适合DID

---

## 步骤 3：方法选择决策

基于数据特征，我们选择 **PSM + DID** 的组合策略：

1. **PSM**：处理截面数据中的选择偏差
2. **DID**：利用面板数据结构，进一步控制时间不变因素

---

## TODO

1. 根据数据特征，评估每种方法的适用性
2. 选择主方法和辅助方法
3. 说明选择理由


In [ ]:
# TODO: 加载数据并检查数据特征
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/ecommerce_618.csv')
print(df.head())
print(df.info())

In [ ]:
# TODO: 评估PSM的适用性
# 检查：1) 协变量丰富度 2) 共同支撑域

from sklearn.linear_model import LogisticRegression

# 估计倾向得分
X = df[['historical_gmv', 'activity_score', 'age', 'membership_level']]
y = df['treatment']

ps_model = LogisticRegression().fit(X, y)
df['propensity_score'] = ps_model.predict_proba(X)[:, 1]

# 检查共同支撑域
print('处理组倾向得分范围:', df[df['treatment']==1]['propensity_score'].describe())
print('对照组倾向得分范围:', df[df['treatment']==0]['propensity_score'].describe())

In [ ]:
# TODO: 评估DID的适用性
# 检查：1) 面板数据结构 2) 平行趋势

# 检查是否有pre/post数据
if 'pre_gmv' in df.columns and 'post_gmv' in df.columns:
    print('有面板数据，适合DID')
else:
    print('无面板数据，不适合DID')

## 方法选择结论

请在此总结你的方法选择：

- **主方法**：?
- **辅助方法**：?
- **选择理由**：?